# Retail Sales & Profitability Analysis
**Internship Task 3 — Exploratory Data Analysis (EDA)**

This notebook analyzes the Sample Superstore dataset. It covers data checks, summary statistics, category/region/discount analysis, monthly trends, and return patterns.

**How to run:** Upload `sample_-_superstore.xlsx` when prompted. If your file has a different name, select that file in the upload dialog.

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
print("Uploaded:", file_name)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from IPython.display import display

xls = pd.ExcelFile(file_name)
print("Sheets:", xls.sheet_names)
orders = pd.read_excel(file_name, sheet_name="Orders")
people = pd.read_excel(file_name, sheet_name="People")
returns = pd.read_excel(file_name, sheet_name="Returns")

orders["Order Date"] = pd.to_datetime(orders["Order Date"], errors="coerce")
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"], errors="coerce")
print("Orders shape:", orders.shape)
display(orders.head())

## 1. Data quality checks

In [ ]:
quality = pd.DataFrame({
    "Missing values": orders.isna().sum(),
    "Data type": orders.dtypes.astype(str)
})
display(quality)
print("Duplicate Row IDs:", orders["Row ID"].duplicated().sum())
print("Unique orders:", orders["Order ID"].nunique())
print("Order date range:", orders["Order Date"].min(), "to", orders["Order Date"].max())

## 2. Prepare analysis fields

In [ ]:
orders["Regional Manager"] = orders["Region"].map(people.set_index("Region")["Regional Manager"])
orders["Shipping Days"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Order Month"] = orders["Order Date"].dt.to_period("M").astype(str)
orders["Profitability Flag"] = np.select(
    [orders["Profit"] > 0, orders["Profit"] < 0],
    ["Profit", "Loss"], default="Break-even"
)
orders["Returned Flag"] = np.where(orders["Order ID"].isin(returns["Order ID"]), "Returned", "Not Returned")
orders["Profit Margin"] = orders["Profit"] / orders["Sales"].replace(0, np.nan)
display(orders.head())

## 3. Key performance indicators

In [ ]:
kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Profit Margin", "Quantity", "Unique Orders", "Negative-profit order lines"],
    "Value": [
        round(orders["Sales"].sum(), 2),
        round(orders["Profit"].sum(), 2),
        round(100 * orders["Profit"].sum() / orders["Sales"].sum(), 2),
        int(orders["Quantity"].sum()),
        int(orders["Order ID"].nunique()),
        int((orders["Profit"] < 0).sum())
    ]
})
display(kpis)

## 4. Category and regional performance

In [ ]:
category = orders.groupby("Category", as_index=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
region = orders.groupby("Region", as_index=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
display(category.sort_values("Sales", ascending=False))
display(region.sort_values("Sales", ascending=False))

category.set_index("Category")[["Sales","Profit"]].plot(kind="bar", figsize=(9,5), title="Sales and Profit by Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

region.set_index("Region")[["Sales","Profit"]].plot(kind="bar", figsize=(9,5), title="Sales and Profit by Region")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Discount and loss analysis

In [ ]:
discount = orders.groupby("Discount", as_index=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"), Lines=("Row ID","count"))
display(discount)
print("Discount levels with negative aggregate profit:")
display(discount.loc[discount["Profit"] < 0].sort_values("Discount"))

subcat = orders.groupby(["Category","Sub-Category"], as_index=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
display(subcat.sort_values("Profit").head(10))
subcat.sort_values("Profit").head(10).plot(x="Sub-Category", y="Profit", kind="barh", figsize=(9,5), title="10 Lowest-Profit Sub-Categories")
plt.tight_layout()
plt.show()

## 6. Monthly trends

In [ ]:
monthly = orders.groupby("Order Month", as_index=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
display(monthly.head())
monthly.plot(x="Order Month", y=["Sales","Profit"], figsize=(12,5), title="Monthly Sales and Profit")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 7. Returns analysis (order-level aggregation avoids counting each order multiple times)

In [ ]:
order_level = orders.groupby("Order ID", as_index=False).agg(
    Sales=("Sales","sum"), Profit=("Profit","sum"), Region=("Region","first"), Category=("Category","first")
)
order_level = order_level.merge(returns, on="Order ID", how="left")
order_level["Return Status"] = order_level["Returned"].fillna("No")
return_summary = order_level.groupby("Return Status", as_index=False).agg(
    Orders=("Order ID","nunique"), Sales=("Sales","sum"), Profit=("Profit","sum")
)
display(return_summary)

## 8. Export analysis tables

In [ ]:
with pd.ExcelWriter("Retail_EDA_Results.xlsx", engine="openpyxl") as writer:
    kpis.to_excel(writer, sheet_name="KPIs", index=False)
    category.to_excel(writer, sheet_name="Category", index=False)
    region.to_excel(writer, sheet_name="Region", index=False)
    discount.to_excel(writer, sheet_name="Discount", index=False)
    subcat.to_excel(writer, sheet_name="Subcategory", index=False)
    monthly.to_excel(writer, sheet_name="Monthly", index=False)
    return_summary.to_excel(writer, sheet_name="Returns", index=False)
files.download("Retail_EDA_Results.xlsx")